##Libraries

In [ ]:
# Import general utilities
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
# from google.colab import drive

# Import TDA utilities
import gudhi
import gudhi.representations
from ripser import Rips
from persim.landscapes import PersLandscapeExact
from persim.landscapes import plot_landscape_simple
from persim.landscapes import PersistenceLandscaper
from persim import plot_diagrams
from tadasets import torus, sphere

# Import Scikit-Learn tools
from sklearn.decomposition import PCA
from sklearn import svm
from sklearn import metrics
from sklearn.model_selection import train_test_split
from mpl_toolkits.mplot3d import Axes3D # Required for 3D plots


##Creating Sample Data: Circle, 1D double Tourus, Torus, and Sphere

In [ ]:
num_points = 100
radius = 1
noise_level = .1

theta = np.linspace(0, 2 * np.pi, num_points)
x_perfect = radius * np.cos(theta)
y_perfect = radius * np.sin(theta)

x_noise = np.random.uniform(-noise_level, noise_level, num_points)
y_noise = np.random.uniform(-noise_level, noise_level, num_points)

x_approx = x_perfect + x_noise
y_approx = y_perfect + y_noise

Circle_data = np.column_stack((x_approx, y_approx))


# Creating Double Torus 2D datacloud points.
num_points_eight = 100
noise_level_eight = 0.1 # Small noise to make it realistic

t = np.linspace(-np.pi, np.pi, num_points_eight)
x_eight = 2*(np.sin(t))
y_eight = 2*(np.cos(t)) * (np.sin(t))


x_noise_eight = np.random.normal(0, noise_level_eight, num_points_eight)
y_noise_eight = np.random.normal(0, noise_level_eight, num_points_eight)

DoubleTorus2D_data = np.column_stack((x_eight + x_noise_eight, y_eight + y_noise_eight))

# Creating Sphere datacloud points with noise.
num_points_sphere = 100
radius_sphere = 1
noise_level_sphere = .1
Sphere_data = sphere(n=num_points_sphere, r=radius_sphere, noise=noise_level_sphere)


# Creating 3D Torus datacloud points with noice.
num_points_torus = 100
c_torus = 1 # Major axis
a_torus = 1 # Minor axis

noise_level_torus = 0.1 # Small noise to make it realistic but retain shape

Torus_data = torus(n=num_points_torus, c=c_torus, a=a_torus, noise=noise_level_torus)

##Visualizing the Data

In [ ]:

fig = plt.figure(figsize=(12, 8))

# Subplot 1: Circular Data (2D)
ax1 = fig.add_subplot(2, 2, 1)
ax1.scatter(Circle_data[:, 0], Circle_data[:, 1], alpha=0.7)
ax1.set_title("Circular Data", fontsize=20) # Increased fontsize to 20

ax1.set_aspect('equal', adjustable='box')
ax1.grid(True)

# Subplot 2: Double Torus 2D Data (Figure-Eight)
ax2 = fig.add_subplot(2, 2, 2)
ax2.scatter(DoubleTorus2D_data[:, 0], DoubleTorus2D_data[:, 1], alpha=0.7)
ax2.set_title("Double Torus 2D Data (Figure-Eight)", fontsize=20) # Increased fontsize to 20
ax2.set_aspect('equal', adjustable='box')
ax2.grid(True)

# Subplot 3: Sphere Data (3D)
ax3 = fig.add_subplot(2, 2, 3, projection='3d')
ax3.scatter(Sphere_data[:, 0], Sphere_data[:, 1], Sphere_data[:, 2], alpha=0.7)
ax3.set_title("Sphere Data (3D)", fontsize=20) # Increased fontsize to 20

# Subplot 4: Torus Data (3D)
ax4 = fig.add_subplot(2, 2, 4, projection='3d')
ax4.scatter(Torus_data[:, 0], Torus_data[:, 1], Torus_data[:, 2], alpha=0.7)
ax4.set_title("Torus Data (3D)", fontsize=20) # Increased fontsize to 20


plt.tight_layout()
plt.show()

##TDA Analysis:

In [ ]:
import pandas as pd
# from google.colab import drive
# drive.mount('/content/drive')

epha2a = pd.read_csv("/workspaces/Cellular-TDA/data/Old Data/centroids Epha2a hom_table.csv")
control = pd.read_csv("/workspaces/Cellular-TDA/data/Old Data/centroids control_table.csv")

x_epha2a = epha2a['Center of the object_0']
y_epha2a = epha2a['Center of the object_1']
z_epha2a = epha2a['Center of the object_2']

x_control = control['Center of the object_0']
y_control = control['Center of the object_1']
z_control = control['Center of the object_2']

epha2_data = np.column_stack((x_epha2a, y_epha2a, z_epha2a))
control_data = np.column_stack((x_control, y_control, z_control))

##Creating Persistance Landscapes Epha2a

In [ ]:
dgm_epha2_data       = Rips(maxdim=2).fit_transform(epha2_data)
dgm_control_data     = Rips(maxdim=2).fit_transform(control_data)


fig = plt.figure(figsize=(10,10))
# Plot Data

#3D plots
ax0 = fig.add_subplot(2, 2, 1, projection='3d')
ax1 = fig.add_subplot(2, 2, 2, projection='3d')

ax0.scatter(epha2_data[:, 0], epha2_data[:, 1], epha2_data[:,2])
ax1.scatter(control_data[:, 0], control_data[:, 1], control_data[:,2])

# Change rotation: 30 degrees elevation, 45 degrees azimuth
ax0.view_init(elev=45, azim=90, roll=0)
ax1.view_init(elev=45, azim=45, roll=0)
# Plot Persistence Diagrams
ax2 = fig.add_subplot(2, 2, 3)
ax3 = fig.add_subplot(2, 2, 4)

plot_diagrams(dgm_control_data, title="Control", ax=ax2)
plot_diagrams(dgm_epha2_data, title="Epha2a", ax=ax3)


# Why are there no voids?

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(15, 7))

# Subplot 1: Epha2a Data with Surface
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_trisurf(epha2_data[:, 0], epha2_data[:, 1], epha2_data[:, 2], cmap='viridis', alpha=0.7)
ax1.scatter(epha2_data[:, 0], epha2_data[:, 1], epha2_data[:, 2], color='red', s=20, alpha=0.9)

ax1.set_title('Epha2a Data with Surface', fontsize=16)
ax1.set_xlabel('X', fontsize=12)
ax1.set_ylabel('Y', fontsize=12)
ax1.set_zlabel('Z', fontsize=12)
ax1.view_init(elev=35, azim=90, roll=0)

# Subplot 2: Control Data with Surface
ax2 = fig.add_subplot(122, projection='3d')
ax2.plot_trisurf(control_data[:, 0], control_data[:, 1], control_data[:, 2], cmap='viridis', alpha=0.7)
ax2.scatter(control_data[:, 0], control_data[:, 1], control_data[:, 2], color='blue', s=20, alpha=0.9)
ax2.set_title('Control Data with Surface', fontsize=16)
ax2.set_xlabel('X', fontsize=12)
ax2.set_ylabel('Y', fontsize=12)
ax2.set_zlabel('Z', fontsize=12)
ax2.view_init(elev=35, azim=45, roll=0)

plt.tight_layout()
plt.show()

###3D Graphs

In [ ]:
try:
    import ipywidgets as widgets
    from ipywidgets import interact, IntSlider
    from IPython.display import display
except ImportError:
    print("ipywidgets not found. Installing...")
    %pip install ipywidgets
    import ipywidgets as widgets
    from ipywidgets import interact, IntSlider
    from IPython.display import display


@interact(elev=IntSlider(min=-90, max=90, step=5, value=35, description='Elevation:'),
          azim=IntSlider(min=0, max=360, step=5, value=90, description='Azimuth:'))
def plot_epha2a_interactive(elev, azim):
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    ax.plot_trisurf(epha2_data[:, 0], epha2_data[:, 1], epha2_data[:, 2], cmap='viridis', alpha=0.7)
    ax.scatter(epha2_data[:, 0], epha2_data[:, 1], epha2_data[:, 2])

    ax.set_title(f'Epha2a Data with Interactive View (Elev: {elev}, Azim: {azim})', fontsize=16)
    ax.set_xlabel('X', fontsize=12)
    ax.set_ylabel('Y', fontsize=12)
    ax.set_zlabel('Z', fontsize=12)

    ax.view_init(elev=elev, azim=azim)
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

@interact(elev=IntSlider(min=-90, max=90, step=5, value=35, description='Elevation:'),
          azim=IntSlider(min=0, max=360, step=5, value=45, description='Azimuth:'))
def plot_control_interactive(elev, azim):
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    ax.plot_trisurf(control_data[:, 0], control_data[:, 1], control_data[:, 2], cmap='viridis', alpha=0.7)
    ax.scatter(control_data[:, 0], control_data[:, 1], control_data[:, 2])

    ax.set_title(f'Control Data with Interactive View (Elev: {elev}, Azim: {azim})', fontsize=16)
    ax.set_xlabel('X', fontsize=12)
    ax.set_ylabel('Y', fontsize=12)
    ax.set_zlabel('Z', fontsize=12)

    ax.view_init(elev=elev, azim=azim)
    plt.show()

In [ ]:
dgm_Circle_data        = Rips(maxdim=1).fit_transform(epha2_data)
dgm_DoubleTorus2D_data = Rips(maxdim=1).fit_transform(DoubleTorus2D_data)
dgm_Sphere_data        = Rips(maxdim=2).fit_transform(Sphere_data)
dgm_Torus_data         = Rips(maxdim=2).fit_transform(Torus_data)

In [ ]:
cloud_data = []
cloud_data.append(dgm_Circle_data)
cloud_data.append(dgm_DoubleTorus2D_data)
cloud_data.append(dgm_Sphere_data)
cloud_data.append(dgm_Torus_data)


n_lambda = 10
remove_legend = True

fig, axs = plt.subplots( len(cloud_data), 3) # 4 Persistence Landscape and three properties ( H0 - H2)
fig.set_size_inches(12, 8)

all_ax = axs.flatten()

# Define column titles
shape         = ["Circle", "Double Torus", "Sphere", "Torus"]
column_titles = ["Connected", "Loops", "Void", "Hyper-Voids"]

for i, dgm_data in enumerate(cloud_data):
  for j in range(len(dgm_data)): # j is the homology dimension (0 for H0, 1 for H1, 2 for H2)
    current_diagram = dgm_data[j]

    # Filter out infinite points as they cannot form a proper landscape function with finite birth/death
    finite_points_in_diagram = np.array([p for p in current_diagram if p[1] != np.inf])

    # Only attempt to plot if there are finite points in the diagram for this homology dimension
    if finite_points_in_diagram.shape[0] == 0:
      continue # Skip plotting for empty or all-infinite diagrams

    plot_landscape_simple(
        PersLandscapeExact(dgm_data, hom_deg=j), # Pass the full list of diagrams
        ax=all_ax[i*3 + j], depth_range = range(n_lambda) )

    # all_ax[i*3+j].set_xlim([0, 2])
    # all_ax[i*3+j].set_ylim([0, 1])
    # Set column titles for the top row only
    if i == 0:
        all_ax[j].set_title(column_titles[j], fontsize=20) # Set title for the first row's subplots

    if ( remove_legend ):
      legend = all_ax[i*3 + j].get_legend()
      if legend:
        legend.remove()

all_ax[2].set_title("Void", fontsize=20)
all_ax[0].set_ylabel("Circle", fontsize=15)
all_ax[3].set_ylabel("Double Torus", fontsize=15)
all_ax[6].set_ylabel("Sphere", fontsize=15)
all_ax[9].set_ylabel("Torus", fontsize=15)
fig.tight_layout() # Adjust layout to provide spacing



In [ ]:
dgm_epha2_data       = Rips(maxdim=2).fit_transform(epha2_data)
dgm_control_data     = Rips(maxdim=2).fit_transform(control_data)


fig = plt.figure(figsize=(10,10))
# Plot Data

#3D plots
ax0 = fig.add_subplot(2, 2, 1, projection='3d')
ax1 = fig.add_subplot(2, 2, 2, projection='3d')

ax0.scatter(epha2_data[:, 0], epha2_data[:, 1], epha2_data[:,2])
ax1.scatter(control_data[:, 0], control_data[:, 1], control_data[:,2])

# Change rotation: 30 degrees elevation, 45 degrees azimuth
ax0.view_init(elev=45, azim=90, roll=0)
ax1.view_init(elev=45, azim=45, roll=0)
# Plot Persistence Diagrams
ax2 = fig.add_subplot(2, 2, 3)
ax3 = fig.add_subplot(2, 2, 4)

plot_diagrams(dgm_control_data, title="Control", ax=ax2)
plot_diagrams(dgm_epha2_data, title="Epha2a", ax=ax3)


pl_epha2 = PersLandscapeExact(dgm_epha2_data, hom_deg=1)
pl_control = PersLandscapeExact(dgm_control_data, hom_deg=1)

fig, axs = plt.subplots(1, 2)
fig.set_size_inches(12, 8)

plot_landscape_simple(pl_epha2, ax=axs[0])
plot_landscape_simple(pl_control, ax=axs[1])

axs[0].get_legend().remove()
axs[1].get_legend().remove()

### Using `PersistenceLandscaper`

First, we instantiate `PersistenceLandscaper`, specifying the homology dimensions we are interested in (`hom_dims`) and the number of landscape functions (`n_landscapes`) we want to compute for each dimension. Then, we use the `fit_transform` method to generate the landscapes from our persistence diagram data.

In [ ]:
from persim.landscapes import PersistenceLandscaper

# Initialize PersistenceLandscaper for H1 (loops)
landscaper = PersistenceLandscaper()

# Fit and transform the epha2_data persistence diagram
# The input should be a list of diagrams, even if it's just one, hence [dgm_epha2_data[1]]
# or if you want to process the full diagram with all dimensions, pass dgm_epha2_data

# Let's process the H1 diagrams from dgm_epha2_data and dgm_control_data

# For epha2_data:
# PersistenceLandscaper expects a list of diagrams (each diagram being a list of (birth, death) pairs).
# dgm_epha2_data is already a list where each element is an array for a homology dimension.
# So, to get H1, we access dgm_epha2_data[1]. If we want to transform multiple, we'd wrap them.

# Example for processing a single diagram (e.g., H1 of epha2_data):
epha2_landscapes_h1 = landscaper.fit_transform([dgm_epha2_data[1]])

# The output will be a numpy array where each row is a flattened landscape function.
# If we transformed a single diagram, the result will have shape (1, num_points_in_landscape * n_landscapes)
# We can reshape it to see individual landscape functions.

print(f"Shape of epha2_landscapes_h1: {epha2_landscapes_h1.shape}")

# You can access individual landscape functions if needed
# For example, to get the first landscape function (eta_1) for the first diagram (epha2_data[1]):
# The landscaper flattens the output. To visualize, you'd typically need to unflatten or use other methods.

# Let's re-initialize to plot one landscape more directly for visualization
# using PersLandscapeExact first to get a single landscape, then show how to work with the output of Landscaper.

# For visualization purposes, let's plot the first landscape (eta_1) from H1 of epha2_data
# This uses PersLandscapeExact directly as shown before, but now you know Landscaper can automate this for many diagrams.

pl_epha2_h1 = PersLandscapeExact(dgm_epha2_data, hom_deg=1)

fig, ax = plt.subplots(figsize=(8, 6))
plot_landscape_simple(pl_epha2_h1, ax=ax, title='Epha2a Data H1 - First Landscape (eta_1)')
ax.get_legend().remove()
plt.show()

# If you wanted to get the vectorized output from PersistenceLandscaper for all H1 diagrams from both datasets:

# First, prepare a list of H1 diagrams
all_h1_diagrams = [dgm_epha2_data[1], dgm_control_data[1]]

# Initialize landscaper (can reuse if parameters are the same)
landscaper_all_h1 = PersistenceLandscaper() # Let's get 3 landscapes

# Transform the list of diagrams
vectorized_landscapes = landscaper_all_h1.fit_transform(all_h1_diagrams)

print(f"\nShape of vectorized landscapes for all H1 diagrams: {vectorized_landscapes.shape}")
print("Each row in this array corresponds to a flattened and concatenated set of landscape functions for one persistence diagram.")
print("This output is directly usable for machine learning algorithms.")